In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# Define a custom Dataset class for our questions.
class ClarificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Function to compute accuracy for evaluation.
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
def train_model(train_dataset_path, model_dir):
    # Load the training TSV file.
    df_train = pd.read_csv(train_dataset_path, sep="\t")

    # Remove duplicate rows based on 'initial_request'
    df_train = df_train.drop_duplicates(subset=["initial_request"])
    print("Rows:", df_train.shape[0], "Columns:", df_train.shape[1])

    # Convert the clarification_need labels (assuming they need to be converted from 1-indexed to 0-indexed)
    df_train["clarification_need"] = df_train["clarification_need"].astype(int) - 1

    # Get the texts and labels from the deduplicated DataFrame.
    train_texts = df_train["initial_request"].tolist()
    train_labels = df_train["clarification_need"].tolist()

    # Optionally, split off a validation set from your training data.
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels, test_size=0.2, random_state=42
    )

    # Initialize the tokenizer.
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    # Create Dataset objects.
    train_dataset = ClarificationDataset(train_texts, train_labels, tokenizer)
    val_dataset = ClarificationDataset(val_texts, val_labels, tokenizer)

    # Load a pre-trained BERT model with a classification head.
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

    # Set up training arguments.
    training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,  # Increased epochs
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,  # Lower learning rate
    warmup_steps=500,    # Added warmup steps
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to=[]  # disables logging to wandb and others
  )


    # Initialize the Trainer.
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Train the model.
    trainer.train()

    # Save the fine-tuned model and tokenizer.
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"Model saved to {model_dir}")
    return tokenizer, model, trainer


In [ ]:
# ------------------ User-Defined Parameters ------------------
# Update these file names/paths as needed.
train_dataset_path = "train.tsv"  # Your training dataset file (TSV)
test_dataset_path = "test.tsv"    # Your testing dataset file (TSV)
test_dataset_with_labels_path = "test_with_labels.tsv"
model_dir = "saved_model"                 # Directory to save/load the model
new_question = "How do I reset my password?"  # Set to None if you don't want to run inference

In [ ]:
# ------------------ Main Execution ------------------
import os
os.environ["WANDB_DISABLED"] = "true"

# If a saved model exists, load it; otherwise, fine-tune on the training dataset.
if os.path.exists(model_dir):
    print("Loading existing model from", model_dir)
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    model = BertForSequenceClassification.from_pretrained(model_dir)
    # Create a dummy TrainingArguments instance (only needed for evaluation via Trainer).
    eval_args = TrainingArguments(output_dir="./results", per_device_eval_batch_size=16)
    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics
    )
else:
    print("No saved model found. Training a new model.")
    tokenizer, model, trainer = train_model(train_dataset_path, model_dir)

In [ ]:
# ------------------ Test Set Evaluation ------------------
if os.path.exists(test_dataset_with_labels_path):
    print("Evaluating on the test dataset:", test_dataset_with_labels_path)
    df_test = pd.read_csv(test_dataset_with_labels_path, sep="\t")
    test_texts = df_test["initial_request"].tolist()
    test_labels = [int(label) - 1 for label in df_test["clarification_need"].tolist()]
    test_dataset = ClarificationDataset(test_texts, test_labels, tokenizer)

    # Evaluate using the Trainer.
    results = trainer.evaluate(test_dataset)
    print("Test Evaluation Results:")
    print(results)

    # Generate predictions for the test dataset.
    predictions_output = trainer.predict(test_dataset)
    preds = predictions_output.predictions.argmax(-1)

    # Generate and display the confusion matrix.
    cm = confusion_matrix(test_labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()
else:
    print("Test dataset not found. Skipping test evaluation.")

In [ ]:
# ------------------ Inference ------------------
# If you wish to perform inference using a new question, set new_question to a non-None value.
if new_question:
    print("Running inference on the new question.")
    text_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, framework="pt")
    prediction = text_classifier(new_question)
    print("Prediction for the input question:")
    print(prediction)
else:
    print("No new question provided. Inference skipped.")

In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

# Define the custom Dataset class (unchanged)
class ClarificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Compute accuracy for evaluation (unchanged)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Updated train_model function
def train_model(train_dataset_path, model_dir):
    # Load and clean the training TSV file
    df_train = pd.read_csv(train_dataset_path, sep="\t")

    # Remove inconsistent labels
    inconsistent = df_train.groupby('initial_request')['clarification_need'].nunique() > 1
    if inconsistent.any():
        conflicting_requests = inconsistent[inconsistent].index
        df_train = df_train[~df_train['initial_request'].isin(conflicting_requests)]
        print(f"Removed {len(conflicting_requests)} conflicting 'initial_request' entries.")

    # Remove duplicates
    df_train = df_train.drop_duplicates(subset=['initial_request'], keep='first')
    print(f"Training set size after deduplication: {len(df_train)}")

    # Check class distribution
    print("Class distribution:")
    print(df_train['clarification_need'].value_counts())

    # Prepare texts and labels
    texts = df_train["initial_request"].tolist()
    labels = [int(label) - 1 for label in df_train["clarification_need"].tolist()]  # 0-based labels

    # Use DistilBERT instead of BERT
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

    # K-Fold Cross-Validation (e.g., 5 folds)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_model = None
    best_accuracy = 0.0

    for fold, (train_idx, val_idx) in enumerate(kf.split(texts)):
        print(f"\nTraining Fold {fold + 1}/5")

        # Split data for this fold
        train_texts = [texts[i] for i in train_idx]
        train_labels = [labels[i] for i in train_idx]
        val_texts = [texts[i] for i in val_idx]
        val_labels = [labels[i] for i in val_idx]

        # Create datasets
        train_dataset = ClarificationDataset(train_texts, train_labels, tokenizer)
        val_dataset = ClarificationDataset(val_texts, val_labels, tokenizer)

        # Optimized training arguments
        training_args = TrainingArguments(
            output_dir=f"./results_fold_{fold}",
            num_train_epochs=5,              # More epochs for small data
            per_device_train_batch_size=8,   # Smaller batch size for stability
            per_device_eval_batch_size=8,
            warmup_steps=10,                 # Gradual learning rate warmup
            weight_decay=0.01,              # Regularization to prevent overfitting
            learning_rate=2e-5,             # Lower LR for fine-tuning
            evaluation_strategy="epoch",
            save_strategy="epoch",
            logging_dir=f"./logs_fold_{fold}",
            logging_steps=10,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            report_to=[]  # No external logging
        )

        # Initialize Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics
        )

        # Train
        trainer.train()

        # Evaluate on validation fold
        eval_results = trainer.evaluate()
        fold_accuracy = eval_results["eval_accuracy"]
        print(f"Fold {fold + 1} accuracy: {fold_accuracy:.4f}")

        # Keep the best model
        if fold_accuracy > best_accuracy:
            best_accuracy = fold_accuracy
            best_model = trainer.model

    # Save the best model
    best_model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"Best model (accuracy: {best_accuracy:.4f}) saved to {model_dir}")

    return tokenizer, best_model, trainer

# Main execution (adjust paths as needed)
os.environ["WANDB_DISABLED"] = "true"
train_dataset_path = "train.tsv"
model_dir = "saved_model"

if os.path.exists(model_dir):
    print("Loading existing model from", model_dir)
    tokenizer = DistilBertTokenizer.from_pretrained(model_dir)
    model = DistilBertForSequenceClassification.from_pretrained(model_dir)
    eval_args = TrainingArguments(output_dir="./results", per_device_eval_batch_size=8)
    trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)
else:
    print("Training new model...")
    tokenizer, model, trainer = train_model(train_dataset_path, model_dir)

# Test evaluation and inference code remains unchanged unless you want tweaks

In [ ]:
# ------------------ Test Set Evaluation ------------------
if os.path.exists(test_dataset_with_labels_path):
    print("Evaluating on the test dataset:", test_dataset_with_labels_path)
    df_test = pd.read_csv(test_dataset_with_labels_path, sep="\t")
    test_texts = df_test["initial_request"].tolist()
    test_labels = [int(label) - 1 for label in df_test["clarification_need"].tolist()]
    test_dataset = ClarificationDataset(test_texts, test_labels, tokenizer)

    # Evaluate using the Trainer.
    results = trainer.evaluate(test_dataset)
    print("Test Evaluation Results:")
    print(results)

    # Generate predictions for the test dataset.
    predictions_output = trainer.predict(test_dataset)
    preds = predictions_output.predictions.argmax(-1)

    # Generate and display the confusion matrix.
    cm = confusion_matrix(test_labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()
else:
    print("Test dataset not found. Skipping test evaluation.")

In [ ]:
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load only 'initial_request' and 'clarification_need' from the TSV
df = pd.read_csv("test_with_labels.tsv", sep="\t", usecols=["initial_request", "clarification_need"])

# Drop duplicates based on these two columns
df_dedup = df.drop_duplicates(subset=["initial_request", "clarification_need"])
print(f"Original rows: {len(df)}, Deduplicated rows: {len(df_dedup)}")

# Define the Dataset class
class TSVDataset(Dataset):
    def __init__(self, dataframe, text_column="initial_request", label_column="clarification_need"):
        self.df = dataframe
        self.texts = self.df[text_column].tolist()
        self.labels = self.df[label_column].tolist()
        self.tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Prepare the test dataset
test_dataset = TSVDataset(df_dedup, text_column="initial_request", label_column="clarification_need")
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Load the saved model from the folder
model = DistilBertForSequenceClassification.from_pretrained("/content/results_fold_3/checkpoint-95")
model.to(device)
model.eval()

# Evaluation function
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    return accuracy, all_preds, all_labels

# Run evaluation
test_accuracy, test_preds, test_labels = evaluate(model, test_loader)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Print some predictions for kicks
for i in range(min(5, len(test_preds))):
    print(f"Text: {test_dataset.texts[i]}")
    print(f"Predicted Label: {test_preds[i]}, True Label: {test_labels[i]}\n")

In [ ]:
!pip install datasets

In [ ]:
import json
import random
import numpy as np
import torch
import os

from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# -----------------------------
# 1. LOAD & PREPARE THE DATASET
# -----------------------------
# We'll assume your JSON data is something like coding_prompts_dataset.json
# Each entry: { "prompt": "...", "clarification_need": 1/2/3/4, "reasoning": "..." }

json_file = "coding_prompts_dataset.json"
with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Keep only prompt + clarification_need
examples = []
for entry in data:
    prompt = entry["prompt"]
    # Original label is 1..4. Convert them to 0..3
    # so DistilBERT can handle them as class indices
    label = int(entry["clarification_need"]) - 1
    examples.append({"text": prompt, "label": label})

# Shuffle the examples (optional, but good practice)
random.shuffle(examples)

# Train/Test split (90% train, 10% test)
split_index = int(0.9 * len(examples))
train_data = examples[:split_index]
test_data = examples[split_index:]

# Convert to HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)
dataset_dict = DatasetDict({"train": train_dataset, "test": test_dataset})

# -----------------------------
# 2. TOKENIZE THE TEXT
# -----------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

encoded_dataset = dataset_dict.map(tokenize_function, batched=True)

# Remove columns we don't need (keeping "label", "input_ids", "attention_mask")
encoded_dataset = encoded_dataset.remove_columns(["text"])
encoded_dataset.set_format("torch")

# -----------------------------
# 3. SETUP THE MODEL
# -----------------------------
# DistilBERT with a classification head. 4 labels total.
num_labels = 4
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

# -----------------------------
# 4. METRICS FOR TRAINING
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# -----------------------------
# 5. TRAINING ARGS
# -----------------------------
# We'll do 10 epochs, save every 2 epochs
# We'll also check accuracy after each epoch
training_args = TrainingArguments(
    output_dir="distilbert_ambiguity_ckpts",
    evaluation_strategy="epoch",        # evaluate after each epoch
    save_strategy="epoch",             # save model after each epoch
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    logging_steps=50,
    load_best_model_at_end=False,       # optional
    metric_for_best_model="accuracy",   # optional
    greater_is_better=True,             # optional
)

# -----------------------------
# 6. TRAIN THE MODEL
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Now, we only want to KEEP the models at epochs 2, 4, 6, 8, 10.
# By default, "save_strategy=epoch" saves a checkpoint after every epoch (1..10).
# You could also manually handle it:
#
# for epoch in range(1, 11):
#    trainer.train(resume_from_checkpoint=...)
#    if epoch % 2 == 0:
#        trainer.save_model(f"epoch_{epoch}_ckpt")
#
# But using the built-in "save_strategy='epoch'" is simpler,
# and you can pick which directory you keep


In [ ]:
# ---------------------------------------------------------
# 7. LOAD A SPECIFIC CHECKPOINT & EVALUATE
#    (Pick the checkpoint you want to test)
# ---------------------------------------------------------
checkpoint_dir = "/content/distilbert_ambiguity_ckpts/checkpoint-7488"
# ^ change to the path of the checkpoint you want to evaluate

print(f"\nLoading model from checkpoint: {checkpoint_dir}")
custom_model = DistilBertForSequenceClassification.from_pretrained(checkpoint_dir)
custom_model.eval()

# Recreate a Trainer with the new model for evaluation
custom_trainer = Trainer(
    model=custom_model,
    tokenizer=tokenizer
)

predictions = custom_trainer.predict(encoded_dataset["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Accuracy
acc = accuracy_score(true_labels, pred_labels)
print(f"\nTest Accuracy with checkpoint {checkpoint_dir}: {acc:.3f}")

# Confusion Matrix
cm = confusion_matrix(true_labels, pred_labels)
print("\nConfusion Matrix:\n", cm)

def plot_confusion_matrix(cm, classes):
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    plt.tight_layout()
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

plot_confusion_matrix(cm, classes=["1","2","3","4"])
plt.show()

# ---------------------------------------------------------
# 8. CLASSIFICATION FUNCTION FOR CUSTOM PROMPTS
# ---------------------------------------------------------
# Use the loaded checkpoint for inference
def classify_prompt(user_prompt):
    inputs = tokenizer(user_prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = custom_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred_label = torch.argmax(probs, dim=-1).item()
    # shift {0,1,2,3} -> {1,2,3,4}
    return pred_label + 1

# ---------------------------------------------------------
# 9. TEST THE CUSTOM PROMPT CLASSIFIER
# ---------------------------------------------------------
example_prompt = "I have a huge codebase in Python but I have no idea why it doesn't work."
result = classify_prompt(example_prompt)
print(f"\nCustom Prompt: {example_prompt}")
print("Predicted Ambiguity (1-4):", result)

In [ ]:
# 1. Define some custom prompts (some detailed, some vague)
custom_prompts = [
    "I'm running a Django server on Ubuntu 20.04 with Gunicorn and Nginx, but I keep getting a 502 Bad Gateway error. Logs show a socket connection refused. What might be misconfigured?",
    "My Java code snippet is throwing an IndexOutOfBoundsException at line 15. The array length is 5, but I'm trying to access index 5. How do I fix this?",
    "How can I optimize this SQL query? SELECT * FROM employees WHERE department_id = 5;",
    "Why won't my code compile?",
    "Help me with my program!",
    "I'm training a PyTorch model on a GPU. It runs for 50 epochs but accuracy doesn't improve beyond 50%. Any ideas?",
    "My React app fails to build. I'm using create-react-app and have some routing issues. Why won't it compile?",
    "In Swift, I'm trying to unwrap an optional but keep getting a runtime crash if it's nil. Here's my code:\n\n```swift\nvar name: String?\nprint(name!)\n```",
    "Node.js server times out. I'm not sure why.",
    "I'm writing a function in C that returns a negative value sometimes, but I expect it to be positive. Something is off."
]

# 2. Classify each prompt
print("=== Custom Prompt Classifications ===\n")
for i, prompt in enumerate(custom_prompts, start=1):
    predicted_label = classify_prompt(prompt)
    print(f"Prompt {i}:")
    print(f"  Text: {prompt}")
    print(f"  Predicted Ambiguity (1–4): {predicted_label}\n")
